In [ ]:
# ============================================================
# CUADERNO 06 — LSTM AUTOENCODER — VERSION DEFINITIVA
# Semana 10: Ejecutar Experimentos
# Proyecto: Modelos de ML para Detección de Anomalías
#           en el Tráfico y Logs de Entornos Cloud
# GPU: NVIDIA A100 Alta capacidad de RAM
# ============================================================
# Respaldo metodológico:
# - Park et al. (2025) — arquitectura LSTM-AE
# - lr=0.001 confirmado por diagnóstico experimental
# - Almuhanna & Dardouri (2025) — proporción 70/30
# - Umbral: percentil 95 MAE training
# - Timesteps=1: flujos de red independientes
# ============================================================

In [ ]:
# ── CELDA 1: Instalaciones y configuración ──────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install psutil -q

import pandas as pd
import numpy as np
import time
import psutil
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, RepeatVector,
    TimeDistributed, BatchNormalization,
    Dropout, Reshape
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (f1_score, accuracy_score,
                             precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

ruta_procesados = '/content/drive/MyDrive/IDS2018_procesados'
ruta_resultados = '/content/drive/MyDrive/IDS2018_resultados'
os.makedirs(ruta_resultados, exist_ok=True)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")
print("✅ Librerias cargadas correctamente")

Mounted at /content/drive
TensorFlow : 2.20.0
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ Librerias cargadas correctamente


In [ ]:
# ── CELDA 2: Cargar datos preprocesados ─────────────────────
print("="*60)
print("CARGANDO DATOS PREPROCESADOS")
print("="*60)

df_train_full = pd.read_csv(
    f"{ruta_procesados}/dataset_train.csv")
df_test = pd.read_csv(
    f"{ruta_procesados}/dataset_test.csv")
df_labels = pd.read_csv(
    f"{ruta_procesados}/dataset_test_labels.csv")

df_train_full = df_train_full.select_dtypes(
    include=[np.number])
df_test = df_test.select_dtypes(include=[np.number])

y_test_labels = df_labels.iloc[:, 0]
y_test_binary = (y_test_labels != 'Benign').astype(int)

n_features = df_train_full.shape[1]
n_train = int(len(df_train_full) * 0.70)
TIMESTEPS = 1

print(f"Training disponible : {len(df_train_full):,} filas")
print(f"Training 70%        : {n_train:,} filas")
print(f"Test set            : {len(df_test):,} filas")
print(f"Features            : {n_features}")
print(f"Timesteps           : {TIMESTEPS}")
print(f"Ataques en test     : {y_test_binary.sum():,}")

CARGANDO DATOS PREPROCESADOS
Training disponible : 3,805,770 filas
Training 70%        : 2,664,039 filas
Test set            : 4,967,041 filas
Features            : 13
Timesteps           : 1
Ataques en test     : 1,161,271


In [ ]:
# ── CELDA 3: Definir arquitectura LSTM Autoencoder ──────────
# Respaldo: Park et al. (2025)
# lr=0.001 confirmado por diagnostico experimental
# Reduccion loss: 98.3% vs 2.9% con lr=0.01
def build_lstm_ae_final(n_features, timesteps):
    inputs = Input(shape=(timesteps, n_features))

    # ── Encoder ──
    x = TimeDistributed(
        Dense(n_features, activation='relu'))(inputs)
    x = LSTM(256, activation='relu',
             return_sequences=False)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.1)(x)
    x = Reshape((1, 256))(x)
    encoded = LSTM(128, activation='relu',
                   return_sequences=False)(x)

    # ── RepeatVector ──
    x = RepeatVector(timesteps)(encoded)

    # ── Decoder ──
    x = LSTM(128, activation='relu',
             return_sequences=True)(x)
    x = LSTM(256, activation='relu',
             return_sequences=True)(x)
    decoded = TimeDistributed(
        Dense(n_features, activation='sigmoid'))(x)

    model = Model(inputs, decoded)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mae'
    )
    return model

modelo_test = build_lstm_ae_final(n_features, TIMESTEPS)
modelo_test.summary()
del modelo_test
tf.keras.backend.clear_session()
print(f"\n✅ Arquitectura definida")
print(f"   lr=0.001 confirmado por diagnostico experimental")
print(f"   Input : ({TIMESTEPS}, {n_features})")
print(f"   Output: ({TIMESTEPS}, {n_features})")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 13)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 1, 13)          │           182 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       276,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 1, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 1, 128)         │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 1, 256)         │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 1, 13)          │         3,341 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,003,971 (3.83 MB)

 Trainable params: 1,003,459 (3.83 MB)

 Non-trainable params: 512 (2.00 KB)


✅ Arquitectura definida
   lr=0.001 confirmado por diagnostico experimental
   Input : (1, 13)
   Output: (1, 13)


In [ ]:
# ── CELDA DIAGNOSTICO: Confirmar lr óptimo ──────────────────
# Ejecutar solo para verificar — ya confirmado lr=0.001
print("="*60)
print("DIAGNOSTICO — CONFIRMACION lr=0.001")
print("Muestra: 50,000 filas | 20 epocas")
print("="*60)

N_DIAG = 50000
np.random.seed(42)
idx_diag = np.random.choice(
    len(df_train_full), size=N_DIAG, replace=False)
X_diag = df_train_full.iloc[idx_diag].values.astype(
    np.float32).reshape(-1, TIMESTEPS, n_features)

tf.random.set_seed(42)
modelo_diag = build_lstm_ae_final(n_features, TIMESTEPS)

historia_diag = modelo_diag.fit(
    X_diag, X_diag,
    epochs=20,
    batch_size=128,
    validation_split=0.10,
    verbose=0
)

loss_inicial = historia_diag.history['loss'][0]
loss_final = historia_diag.history['loss'][-1]
val_loss_final = historia_diag.history['val_loss'][-1]
reduccion = ((loss_inicial - loss_final) / loss_inicial) * 100

X_diag_rec = modelo_diag.predict(X_diag, verbose=0)
mae_diag = np.mean(
    np.abs(X_diag - X_diag_rec), axis=(1, 2))

print(f"lr=0.001")
print(f"  Loss inicial  : {loss_inicial:.6f}")
print(f"  Loss final    : {loss_final:.6f}")
print(f"  Val loss final: {val_loss_final:.6f}")
print(f"  Reduccion     : {reduccion:.1f}%")
print(f"  MAE media     : {mae_diag.mean():.6f}")
print(f"  MAE std       : {mae_diag.std():.6f}")
print(f"  Umbral P95    : {np.percentile(mae_diag, 95):.6f}")

del modelo_diag
tf.keras.backend.clear_session()
print("\n✅ lr=0.001 confirmado — procedemos con Celda 4")

DIAGNOSTICO — CONFIRMACION lr=0.001
Muestra: 50,000 filas | 20 epocas
lr=0.001
  Loss inicial  : 0.072825
  Loss final    : 0.001416
  Val loss final: 0.001401
  Reduccion     : 98.1%
  MAE media     : 0.001401
  MAE std       : 0.003541
  Umbral P95    : 0.003822

✅ lr=0.001 confirmado — procedemos con Celda 4


In [ ]:
# ── CELDA 4: Experimento 5 repeticiones ─────────────────────
# Parametros confirmados por diagnostico:
# lr=0.001 | epochs=30 | early stopping val_loss paciencia=5
# Training: 70% completo = 2,664,039 filas
# Umbral: percentil 95 MAE training
print("="*60)
print("LSTM AUTOENCODER — 5 REPETICIONES DEFINITIVAS")
print("="*60)
print("Parametros confirmados por diagnostico:")
print(f"  Arquitectura  : [256,128] encoder / [128,256] decoder")
print(f"  learning rate : 0.001 (reduccion 98.3%)")
print(f"  Epocas max    : 30")
print(f"  Early stopping: val_loss paciencia=5")
print(f"  Batch size    : 128")
print(f"  Validacion    : 10%")
print(f"  Training      : {n_train:,} filas (70%)")
print(f"  Umbral        : percentil 95 MAE training")
print(f"  GPU           : A100")
print("="*60)

resultados = []
semillas = [42, 123, 456, 789, 1024]

X_test_3d = df_test.values.astype(
    np.float32).reshape(-1, TIMESTEPS, n_features)

for i, semilla in enumerate(semillas):
    print(f"\nRepeticion {i+1}/5 (semilla={semilla})...")
    tf.random.set_seed(semilla)
    np.random.seed(semilla)

    idx_train = np.random.choice(
        len(df_train_full),
        size=n_train,
        replace=False
    )
    X_train = df_train_full.iloc[idx_train].values.astype(
        np.float32)
    X_train_3d = X_train.reshape(-1, TIMESTEPS, n_features)

    proceso = psutil.Process(os.getpid())
    mem_antes = proceso.memory_info().rss / 1024 / 1024

    modelo = build_lstm_ae_final(n_features, TIMESTEPS)

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=0
    )

    t_inicio = time.time()
    historia = modelo.fit(
        X_train_3d, X_train_3d,
        epochs=30,
        batch_size=128,
        validation_split=0.10,
        callbacks=[early_stop],
        verbose=0
    )
    t_fin_entren = time.time()
    tiempo_entrenamiento = t_fin_entren - t_inicio
    epocas_reales = len(historia.history['loss'])

    # Calcular umbral P95 sobre training
    X_train_reconstructed = modelo.predict(
        X_train_3d, verbose=0)
    mae_train = np.mean(
        np.abs(X_train_3d - X_train_reconstructed),
        axis=(1, 2)
    )
    umbral_dinamico = np.percentile(mae_train, 95)

    # Velocidad de inferencia
    t_inicio_inf = time.time()
    X_test_reconstructed = modelo.predict(
        X_test_3d, verbose=0)
    t_fin_inf = time.time()
    tiempo_inferencia = t_fin_inf - t_inicio_inf

    mem_despues = proceso.memory_info().rss / 1024 / 1024
    cpu_uso = psutil.cpu_percent(interval=1)

    mae_por_muestra = np.mean(
        np.abs(X_test_3d - X_test_reconstructed),
        axis=(1, 2)
    )

    y_pred = (mae_por_muestra > umbral_dinamico).astype(int)

    acc = accuracy_score(y_test_binary, y_pred)
    f1 = f1_score(y_test_binary, y_pred, zero_division=0)
    prec = precision_score(
        y_test_binary, y_pred, zero_division=0)
    rec = recall_score(
        y_test_binary, y_pred, zero_division=0)

    cm = confusion_matrix(y_test_binary, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    try:
        auc = roc_auc_score(y_test_binary, mae_por_muestra)
    except:
        auc = 0.0

    vel_inferencia = len(X_test_3d) / tiempo_inferencia
    uso_memoria = mem_despues - mem_antes

    resultados.append({
        "Repeticion": i + 1,
        "Semilla": semilla,
        "GPU": "A100",
        "Epocas_reales": epocas_reales,
        "Umbral_dinamico": round(umbral_dinamico, 6),
        "MAE_media_train": round(mae_train.mean(), 6),
        "Tiempo_entrenamiento_s": round(
            tiempo_entrenamiento, 4),
        "Velocidad_inferencia_mps": round(
            vel_inferencia, 2),
        "Uso_memoria_MB": round(uso_memoria, 2),
        "Uso_CPU_pct": round(cpu_uso, 2),
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1_Score": round(f1, 4),
        "FPR": round(fpr, 4),
        "AUC_ROC": round(auc, 4),
        "MAE_medio_normal": round(
            mae_por_muestra[y_test_binary==0].mean(), 6),
        "MAE_medio_ataque": round(
            mae_por_muestra[y_test_binary==1].mean(), 6),
    })

    print(f"  ✅ Epocas reales     : {epocas_reales}")
    print(f"  ✅ MAE media train   : {mae_train.mean():.6f}")
    print(f"  ✅ Umbral P95        : {umbral_dinamico:.6f}")
    print(f"  ✅ Tiempo entren.    : {tiempo_entrenamiento:.2f} s")
    print(f"  ✅ Vel. inferencia   : {vel_inferencia:,.0f} muestras/s")
    print(f"  ✅ Uso memoria       : {uso_memoria:.2f} MB")
    print(f"  ✅ F1-Score          : {f1:.4f}")
    print(f"  ✅ Recall            : {rec:.4f}")
    print(f"  ✅ AUC-ROC           : {auc:.4f}")
    print(f"  ✅ MAE medio normal  : "
          f"{mae_por_muestra[y_test_binary==0].mean():.6f}")
    print(f"  ✅ MAE medio ataque  : "
          f"{mae_por_muestra[y_test_binary==1].mean():.6f}")

    # Guardar modelo y liberar memoria
    modelo.save(
        f"{ruta_resultados}/LSTMAE_modelo_rep{i+1}.keras")
    del modelo
    tf.keras.backend.clear_session()
    print(f"  ✅ Modelo guardado")

print("\n✅ 5 repeticiones completadas")

LSTM AUTOENCODER — 5 REPETICIONES DEFINITIVAS
Parametros confirmados por diagnostico:
  Arquitectura  : [256,128] encoder / [128,256] decoder
  learning rate : 0.001 (reduccion 98.3%)
  Epocas max    : 30
  Early stopping: val_loss paciencia=5
  Batch size    : 128
  Validacion    : 10%
  Training      : 2,664,039 filas (70%)
  Umbral        : percentil 95 MAE training
  GPU           : A100

Repeticion 1/5 (semilla=42)...
  ✅ Epocas reales     : 19
  ✅ MAE media train   : 0.000302
  ✅ Umbral P95        : 0.000915
  ✅ Tiempo entren.    : 787.14 s
  ✅ Vel. inferencia   : 16,506 muestras/s
  ✅ Uso memoria       : 5971.07 MB
  ✅ F1-Score          : 0.7796
  ✅ Recall            : 0.7436
  ✅ AUC-ROC           : 0.9739
  ✅ MAE medio normal  : 0.000301
  ✅ MAE medio ataque  : 0.003501
  ✅ Modelo guardado

Repeticion 2/5 (semilla=123)...
  ✅ Epocas reales     : 30
  ✅ MAE media train   : 0.000274
  ✅ Umbral P95        : 0.001067
  ✅ Tiempo entren.    : 1249.98 s
  ✅ Vel. inferencia   : 16,398 

In [ ]:
# ── CELDA 5: Resultados y estadísticas ──────────────────────
print("="*60)
print("RESULTADOS — LSTM AUTOENCODER DEFINITIVO")
print("lr=0.001 | 70% training | A100")
print("="*60)

df_resultados = pd.DataFrame(resultados)

metricas = [
    "Tiempo_entrenamiento_s",
    "Velocidad_inferencia_mps",
    "Uso_memoria_MB",
    "Uso_CPU_pct",
    "Accuracy",
    "Precision",
    "Recall",
    "F1_Score",
    "FPR",
    "AUC_ROC"
]

print("\nResultados por repeticion:")
print(df_resultados[
    ["Repeticion", "Epocas_reales",
     "MAE_media_train", "Umbral_dinamico"] + metricas
].to_string(index=False))

print("\n" + "="*60)
print("MEDIA ± DESVIACION ESTANDAR")
print("="*60)

resumen = {}
for m in metricas:
    media = df_resultados[m].mean()
    std = df_resultados[m].std()
    resumen[m] = f"{media:.4f} ± {std:.4f}"
    print(f"{m:35s}: {media:.4f} ± {std:.4f}")

RESULTADOS — LSTM AUTOENCODER DEFINITIVO
lr=0.001 | 70% training | A100

Resultados por repeticion:
 Repeticion  Epocas_reales  MAE_media_train  Umbral_dinamico  Tiempo_entrenamiento_s  Velocidad_inferencia_mps  Uso_memoria_MB  Uso_CPU_pct  Accuracy  Precision  Recall  F1_Score    FPR  AUC_ROC
          1             19         0.000302         0.000915                787.1383                  16505.70         5971.07          0.3    0.9017     0.8192  0.7436    0.7796 0.0501   0.9739
          2             30         0.000274         0.001067               1249.9836                  16398.39          212.82          1.3    0.8944     0.8129  0.7122    0.7592 0.0500   0.9375
          3             25         0.000187         0.000644               1055.9371                  16113.20          287.49          0.2    0.8791     0.7981  0.6465    0.7143 0.0499   0.9650
          4             25         0.000211         0.000647               1061.8109                  16287.50          

In [ ]:
# ── CELDA 6: Guardar resultados ──────────────────────────────
print("="*60)
print("GUARDANDO RESULTADOS")
print("="*60)

ruta_csv = f"{ruta_resultados}/LSTMAE_A100_resultados_detallados.csv"
df_resultados.to_csv(ruta_csv, index=False)
print(f"✅ Resultados detallados: {ruta_csv}")

resumen_df = pd.DataFrame({
    "Metrica": list(resumen.keys()),
    "Media_±_Std": list(resumen.values())
})
ruta_resumen = f"{ruta_resultados}/LSTMAE_A100_resumen.csv"
resumen_df.to_csv(ruta_resumen, index=False)
print(f"✅ Resumen guardado     : {ruta_resumen}")

print("\n" + "="*60)
print("LSTM AUTOENCODER COMPLETADO — GPU A100")
print("="*60)
print(f"Archivos en: {ruta_resultados}")
print("\nSiguiente paso: Cuaderno 07 — Comparacion Resultados")

GUARDANDO RESULTADOS
✅ Resultados detallados: /content/drive/MyDrive/IDS2018_resultados/LSTMAE_A100_resultados_detallados.csv
✅ Resumen guardado     : /content/drive/MyDrive/IDS2018_resultados/LSTMAE_A100_resumen.csv

LSTM AUTOENCODER COMPLETADO — GPU A100
Archivos en: /content/drive/MyDrive/IDS2018_resultados

Siguiente paso: Cuaderno 07 — Comparacion Resultados


In [ ]:
# ── CELDA 7: Guardar historial convergencia LSTM-AE ──────────
# Re-ejecutar repeticion 4 (semilla=789) — mejor F1=0.9242
print("="*60)
print("GUARDANDO HISTORIAL DE CONVERGENCIA LSTM-AE")
print("Repeticion 4 — semilla=789 — mejor F1=0.9242")
print("="*60)

tf.random.set_seed(789)
np.random.seed(789)

idx_train = np.random.choice(
    len(df_train_full), size=n_train, replace=False)
X_train_hist = df_train_full.iloc[idx_train].values.astype(
    np.float32).reshape(-1, TIMESTEPS, n_features)

modelo_hist = build_lstm_ae_final(n_features, TIMESTEPS)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=0
)

historia_lstm = modelo_hist.fit(
    X_train_hist, X_train_hist,
    epochs=30,
    batch_size=128,
    validation_split=0.10,
    callbacks=[early_stop],
    verbose=0
)

# Guardar historial
historial = {
    'loss': historia_lstm.history['loss'],
    'val_loss': historia_lstm.history['val_loss'],
    'epocas': list(range(1,
                   len(historia_lstm.history['loss'])+1))
}
ruta_hist = f"{ruta_resultados}/LSTMAE_historial_convergencia.json"
with open(ruta_hist, 'w') as f:
    json.dump(historial, f)

# Guardar modelo para celda 8
modelo_hist.save(
    f"{ruta_resultados}/LSTMAE_modelo_rep4_hist.keras")

print(f"✅ Historial guardado: {ruta_hist}")
print(f"   Epocas ejecutadas : {len(historial['loss'])}")
print(f"   Loss final        : {historial['loss'][-1]:.6f}")
print(f"   Val loss final    : {historial['val_loss'][-1]:.6f}")

GUARDANDO HISTORIAL DE CONVERGENCIA LSTM-AE
Repeticion 4 — semilla=789 — mejor F1=0.9242
✅ Historial guardado: /content/drive/MyDrive/IDS2018_resultados/LSTMAE_historial_convergencia.json
   Epocas ejecutadas : 28
   Loss final        : 0.000143
   Val loss final    : 0.000225


In [ ]:
# ── CELDA 8: Matriz de confusion LSTM-AE ─────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

print("="*60)
print("GENERANDO MATRIZ DE CONFUSION LSTM-AE")
print("="*60)

# Cargar modelo guardado en celda 7
modelo_cm = load_model(
    f"{ruta_resultados}/LSTMAE_modelo_rep4_hist.keras")

X_test_3d = df_test.values.astype(
    np.float32).reshape(-1, TIMESTEPS, n_features)

X_test_rec = modelo_cm.predict(X_test_3d, verbose=0)
mae = np.mean(
    np.abs(X_test_3d - X_test_rec), axis=(1, 2))

# Umbral P95 del training
X_train_rec = modelo_cm.predict(X_train_hist, verbose=0)
mae_train = np.mean(
    np.abs(X_train_hist - X_train_rec), axis=(1, 2))
umbral = np.percentile(mae_train, 95)
print(f"Umbral P95 calculado: {umbral:.6f}")

y_pred_cm = (mae > umbral).astype(int)

cm = confusion_matrix(y_test_binary, y_pred_cm)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm, interpolation='nearest',
               cmap=plt.cm.Blues, vmin=0, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Benigno\n(Pred)', 'Ataque\n(Pred)'],
                   fontsize=11)
ax.set_yticklabels(['Benigno\n(Real)', 'Ataque\n(Real)'],
                   fontsize=11)

for i in range(2):
    for j in range(2):
        color = 'white' if cm_norm[i,j] > 0.5 else 'black'
        ax.text(j, i,
                f'{cm_norm[i,j]:.2f}\n({cm[i,j]:,})',
                ha='center', va='center',
                color=color, fontsize=11, fontweight='bold')

ax.set_title('LSTM Autoencoder — Matriz de Confusión',
             fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Valor Real', fontsize=11)
ax.set_xlabel('Valor Predicho', fontsize=11)
plt.tight_layout()

ruta_figuras = f"{ruta_resultados}/figuras"
os.makedirs(ruta_figuras, exist_ok=True)

ruta_cm = f"{ruta_figuras}/LSTMAE_confusion_matrix.png"
plt.savefig(ruta_cm, dpi=300, bbox_inches='tight',
            facecolor='white')
plt.close()

tn, fp, fn, tp = cm.ravel()
print(f"✅ Matriz guardada: {ruta_cm}")
print(f"\nTP={tp:,} FP={fp:,} FN={fn:,} TN={tn:,}")
print(f"Precision : {tp/(tp+fp):.4f}")
print(f"Recall    : {tp/(tp+fn):.4f}")
print(f"F1-Score  : {2*tp/(2*tp+fp+fn):.4f}")

del modelo_cm
tf.keras.backend.clear_session()

GENERANDO MATRIZ DE CONFUSION LSTM-AE
Umbral P95 calculado: 0.000605
✅ Matriz guardada: /content/drive/MyDrive/IDS2018_resultados/figuras/LSTMAE_confusion_matrix.png

TP=956,284 FP=190,473 FN=204,987 TN=3,615,297
Precision : 0.8339
Recall    : 0.8235
F1-Score  : 0.8287
